In [1]:
#use "../lib/nb.ml";;
load_css ()

- : unit = ()
Findlib has been successfully loaded. Additional directives:
  #require "package";;      to load a package
  #list;;                   to list the available packages
  #camlp4o;;                to load camlp4 (standard syntax)
  #camlp4r;;                to load camlp4 (revised syntax)
  #predicates "p,q,...";;   to set these predicates
  Topfind.reset();;         to force that packages will be reloaded
  #thread;;                 to enable threads

- : unit = ()
- : unit = ()
val lib_dir : string = "../lib"
val loaded_libs : (string, unit) Hashtbl.t = <abstr>
val load_lib : string -> unit = <fun>


- : unit = ()


# A Recursive Parser for Arithmetic Expressions

In this notebook we implement a simple *recursive descend* parser for arithmetic expressions.
This parser will implement the following grammar:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{product}\;\;\mathrm{exprRest}            \\[0.2cm]
  \mathrm{exprRest}    & \rightarrow & \texttt{'+'} \;\;\mathrm{product}\;\;\mathrm{exprRest}   \\
                       & \mid        & \texttt{'-'} \;\;\mathrm{product}\;\;\mathrm{exprRest}   \\
                       & \mid        & \lambda                                      \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{factor}\;\;\mathrm{productRest}          \\[0.2cm]
  \mathrm{productRest} & \rightarrow & \texttt{'*'} \;\;\mathrm{factor}\;\;\mathrm{productRest} \\
                       & \mid        & \texttt{'/'} \;\;\mathrm{factor}\;\;\mathrm{productRest} \\
                       & \mid        & \lambda                                      \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}                \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

## Type Checking

The functions in this notebook carry *type annotations*.  In OCaml, these annotations are checked by the
compiler before a cell is executed, so there is no need for a separate type checker like *basedpyright*.
A cell that contains a type error is not executed at all.

## Implementing a Scanner

We implement a scanner with the help of the module `Re` from the file `../lib/regex.ml`, which offers the
functions of Python's module `re` on top of the library PCRE2.

In [2]:
load_lib "regex.ml"

- : unit = ()


The function `tokenize` receives a string `s` as argument and returns a list of tokens.
The string `s` is supposed to represent an arithmetical expression. 

**Note:** 
1. We need to set the flag `VERBOSE` in our call of the function `Re.findall`
   below because otherwise we are not able to format the regular expression `lexSpec` the way 
   we have done it.  Furthermore, we wouldn't have been able to add comments inside the regular expression.
2. The last alternative of the regular expression `lexSpec` is empty.  Therefore, `Re.findall` also returns
   empty strings, which are skipped together with the blanks and tabs.

In [3]:
let tokenize (s : string) : string list =
  (* Transform the string s into a list of tokens.  The string s
     is supposed to represent an arithmetic expression. *)
  let lexSpec = {|[ \t]+        |  # blanks and tabs
                  [1-9][0-9]*|0 |  # numbers
                  [-+*/()]      |  # arithmetical operators and parentheses
               |}
  in
  let tokenList = Re.findall ~flags:Re.[VERBOSE] lexSpec s in
  List.filter (fun token -> not (token = "" || token.[0] = ' ' || token.[0] = '\t')) tokenList  (* skip blanks and tabs *)

val tokenize : string -> string list = <fun>


In [4]:
tokenize "123 + (234 +  345 - 2**0)/7"

- : string list =
["123"; "+"; "("; "234"; "+"; "345"; "-"; "2"; "*"; "*"; "0"; ")"; "/"; "7"]


## Implementing the Recursive Descend Parser

The function `parse` takes a string `s` as input and parses this string according to the recursive grammar
shown above.  The function returns the floating point number that results from evaluation the expression given in the string `s`. 

The parsing functions call each other.  For example, `parse` calls `parseExpr`, which is only defined below,
and `parseFactor` calls `parseExpr` again.  In OCaml, a function can only use functions that have already been
defined, unless all of them are defined together using `let rec ... and ...`.  In order to keep every function
in a cell of its own, we use *forward references*: the next cell creates references, i.e. mutable boxes, that
will later hold the parsing functions.  `!parseExpr_ref` is the function that is currently stored in the box
`parseExpr_ref`.  After a function has been defined, it is stored in its box with the operator `:=`.

In [5]:
let not_yet_defined _ = failwith "not yet defined"

let parseExpr_ref        : (string list -> float * string list) ref          = ref not_yet_defined
let parseExprRest_ref    : (float -> string list -> float * string list) ref = ref not_yet_defined
let parseProduct_ref     : (string list -> float * string list) ref          = ref not_yet_defined
let parseProductRest_ref : (float -> string list -> float * string list) ref = ref not_yet_defined
let parseFactor_ref      : (string list -> float * string list) ref          = ref not_yet_defined

let show_tokens (tl : string list) : string = "[" ^ String.concat ", " (List.map repr tl) ^ "]"

val not_yet_defined : 'a -> 'b = <fun>


val parseExpr_ref : (string list -> float * string list) ref =
  {contents = <fun>}


val parseExprRest_ref : (float -> string list -> float * string list) ref =
  {contents = <fun>}


val parseProduct_ref : (string list -> float * string list) ref =
  {contents = <fun>}


val parseProductRest_ref : (float -> string list -> float * string list) ref =
  {contents = <fun>}


val parseFactor_ref : (string list -> float * string list) ref =
  {contents = <fun>}


val show_tokens : string list -> string = <fun>


In [6]:
let parse (s : string) : float =
  let tl = tokenize s in
  match !parseExpr_ref tl with
  | result, [] -> result
  | _ -> failwith (Printf.sprintf "Parse Error: could not parse %s" (show_tokens tl))

val parse : string -> float = <fun>


The function `parseExpr` implements the following grammar rule:
$$ \mathrm{expr} \rightarrow \;\mathrm{product}\;\;\mathrm{exprRest} $$
It takes a token list `tl` as its input and returns a pair of the form `(value, rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `tl` and
- `rest` is a list of those tokens that have not been consumed during the parse process. 

In [7]:
let parseExpr (tl : string list) : float * string list =
  let product, rest = !parseProduct_ref tl in
  !parseExprRest_ref product rest
;;
parseExpr_ref := parseExpr

val parseExpr : string list -> float * string list = <fun>


- : unit = ()


The function `parseExprRest` implements the following grammar rules:
$$
  \begin{eqnarray*}
  \mathrm{exprRest}    & \rightarrow & \texttt{'+'} \;\;\mathrm{product}\;\;\mathrm{exprRest}   \\
                       & \mid        & \texttt{'-'} \;\;\mathrm{product}\;\;\mathrm{exprRest}   \\
                       & \mid        & \;\lambda                                     \\[0.2cm]
  \end{eqnarray*}
$$
It takes two arguments:
- `sum` is the value that has already been computed from the tokens parsed so far,
- `tl` is the list of tokens that still need to be consumed.

It returns a pair of the form `(value, rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `tl` and
- `rest` is a list of those tokens that have not been consumed during the parse process. 

In [8]:
let rec parseExprRest (sum : float) (tl : string list) : float * string list =
  match tl with
  | [] -> sum, []
  | "+" :: rl ->
      let product, rest = !parseProduct_ref rl in
      parseExprRest (sum +. product) rest
  | "-" :: rl ->
      let product, rest = !parseProduct_ref rl in
      parseExprRest (sum -. product) rest
  | _ -> sum, tl
;;
parseExprRest_ref := parseExprRest

val parseExprRest : float -> string list -> float * string list = <fun>


- : unit = ()


The function `parseProduct` implements the following grammar rule:
$$ \mathrm{product} \rightarrow \;\mathrm{factor}\;\;\mathrm{productRest} $$

It takes one argument:
- `tl` is the list of tokens that need to be consumed.

It returns a pair of the form `(value, rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `tl` and
- `rest` is a list of those tokens that have not been consumed while trying to parse a product.

In [9]:
let parseProduct (tl : string list) : float * string list =
  let factor, rest = !parseFactor_ref tl in
  !parseProductRest_ref factor rest
;;
parseProduct_ref := parseProduct

val parseProduct : string list -> float * string list = <fun>


- : unit = ()


The function `parseProductRest` implements the following grammar rules:
$$
  \begin{eqnarray*}
  \mathrm{productRest} & \rightarrow & \texttt{'*'} \;\;\mathrm{factor}\;\;\mathrm{productRest} \\
                       & \mid        & \texttt{'/'} \;\;\mathrm{factor}\;\;\mathrm{productRest} \\
                       & \mid        & \;\lambda    \\                                  
  \end{eqnarray*}
$$

It takes two arguments:
- `product` is the value that has already been parsed,
- `tl` is the list of tokens that still need to be consumed.

It returns a pair of the form `(value, rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `tl` and
- `rest` is a list of those tokens that have not been consumed while trying to parse the rest of a product.

In [10]:
let rec parseProductRest (product : float) (tl : string list) : float * string list =
  match tl with
  | [] -> product, []
  | "*" :: rl ->
      let factor, rest = !parseFactor_ref rl in
      parseProductRest (product *. factor) rest
  | "/" :: rl ->
      let factor, rest = !parseFactor_ref rl in
      parseProductRest (product /. factor) rest
  | _ -> product, tl
;;
parseProductRest_ref := parseProductRest

val parseProductRest : float -> string list -> float * string list = <fun>


- : unit = ()


The function `parseFactor` implements the following grammar rules:
$$
  \begin{eqnarray*}
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}                \\
                       & \mid        & \;\texttt{NUMBER} 
  \end{eqnarray*}
$$

It takes one argument:
- `tl` is the list of tokens that still need to be consumed.

It returns a pair of the form `(value, rest)` where
- `value` is the result of evaluating the arithmetical expression
  that is represented by `tl` and
- `rest` is a list of those tokens that have not been consumed while trying to parse a factor.

In [11]:
let parseFactor (tl : string list) : float * string list =
  match tl with
  | "(" :: rl -> (
      match parseExpr rl with
      | expr, ")" :: rest -> expr, rest
      | _ -> failwith "Parse Error: expected \")\"")
  | t :: rest -> float_of_string t, rest
  | [] -> failwith "Parse Error: unexpected end of input"
;;
parseFactor_ref := parseFactor

val parseFactor : string list -> float * string list = <fun>


- : unit = ()


## Testing

The function `test` compares the result of `parse` with the result of Python's function `eval`, which evaluates
a string as Python code.  OCaml has no such function.  However, this notebook runs in the OCaml *toplevel*,
which is able to execute OCaml code that is given as a string.  The function `eval` below translates the
arithmetic expression into OCaml syntax for floating point numbers, e.g. `2-3` becomes `2.-.3.`, and lets the
toplevel compute the value.  `Toploop.parse_toplevel_phrase` parses a string into OCaml syntax and
`Toploop.execute_phrase` executes it.  The result is stored in the reference `eval_result`.

In [12]:
let eval_result = ref 0.0

let eval (s : string) : float =
  let ocaml = Re.sub_fn {|[0-9]+|[-+*/]|} (fun m -> Re.group m 0 ^ ".") s in
  let phrase = !Toploop.parse_toplevel_phrase (Lexing.from_string ("eval_result := " ^ ocaml ^ ";;")) in
  if not (Toploop.execute_phrase false Format.err_formatter phrase) then failwith ("eval: " ^ s);
  !eval_result

let test (s : string) : float =
  let r1 = parse s in
  let r2 = eval s in
  assert (r1 = r2);
  r1

val eval_result : float ref = {contents = 0.0}


val eval : string -> float = <fun>


val test : string -> float = <fun>


In [13]:
test "11+22*(33-44)/(5-10*5/(4-3))"

- : float = 16.37777777777778


In [14]:
test "0*11+22*(33-44)/(5-10*5/(4-3))"

- : float = 5.377777777777778


In [15]:
test "2-3-4"

- : float = -5.0
